## Import Package

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from keras.utils import to_categorical
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense

## Import Datasets

In [22]:
(x_train_image ,y_train_label) ,(x_test_image ,y_test_label) = mnist.load_data()

## Confirmed Data Collected

In [23]:
print("x_train_image:" ,x_train_image.shape)
print("y_train_label:" ,y_train_label.shape)

x_train_image: (60000, 28, 28)
y_train_label: (60000,)


In [24]:
print("x_test_image:" ,x_test_image.shape)
print("y_test_label:" ,y_test_label.shape)

x_test_image: (10000, 28, 28)
y_test_label: (10000,)


In [25]:
x_Train = x_train_image.reshape(60000 ,784).astype('float32')
x_Test = x_test_image.reshape(10000 ,784).astype('float32')

In [26]:
print("x_Train:" ,x_Train.shape)
print("x_Test:" ,x_Test.shape)

x_Train: (60000, 784)
x_Test: (10000, 784)


## Normalize

In [27]:
x_Train_normalize = x_Train / 255
x_Test_normalize = x_Test / 255

## One Hot Encoding

In [28]:
y_Train_OneHot = to_categorical(y_train_label)
y_Test_OneHot = to_categorical(y_test_label)

## Model

In [29]:
model = Sequential()

In [30]:
model.add(Dense(units = 256,
                input_dim = 784,
                kernel_initializer = "normal",
                activation = "relu"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [31]:
model.add(Dense(units = 10,
                kernel_initializer="normal",
                activation="softmax"))

In [32]:
print(model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 203,530 (795.04 KB)

 Trainable params: 203,530 (795.04 KB)

 Non-trainable params: 0 (0.00 B)

None


In [33]:
model.compile(loss = "categorical_crossentropy" ,
              optimizer = "adam",
              metrics = ["accuracy"])

In [34]:
train_history = model.fit(x = x_Train_normalize,
                          y = y_Train_OneHot,
                          validation_split = 0.2,
                          epochs = 10,
                          batch_size = 200,
                          verbose = 0)

In [35]:
scores = model.evaluate(x_Test_normalize ,y_Test_OneHot ,batch_size = 1)
print("Accrucy =", scores[1])

10000/10000 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step - accuracy: 0.9788 - loss: 0.0712
Accrucy = 0.9787999987602234


In [36]:
def statistics(x_normalize_data ,y_label_data ,batch_size=1):
  pre = model.predict(x_normalize_data ,batch_size)
  prediction = np.argmax(pre ,axis=1)
  conf_matrix = np.zeros((10, 10), dtype=int)

  for true_idx, pred_idx in zip(y_label_data, prediction):
    conf_matrix[true_idx, pred_idx] += 1

  return prediction ,conf_matrix

In [40]:
prediction ,conf_matrix = statistics(x_Test_normalize ,y_test_label)

10000/10000 ━━━━━━━━━━━━━━━━━━━━ 14s 1ms/step


In [41]:
print("Conf_Matrix:")
print(conf_matrix)

Conf_Matrix:
[[ 966    0    4    1    1    2    2    1    3    0]
 [   0 1124    4    0    0    1    2    0    4    0]
 [   2    0 1012    1    1    0    2    7    7    0]
 [   2    0    7  989    0    3    0    4    4    1]
 [   1    0    6    1  955    1    4    3    0   11]
 [   2    0    0    4    0  878    3    1    3    1]
 [   6    3    2    1    3    7  933    0    3    0]
 [   0    3    8    4    0    0    0 1008    2    3]
 [   2    0    5    4    2    4    1    4  950    2]
 [   3    4    0    8    7    3    0    9    2  973]]


In [42]:
print("Max combin: ",end="")

max_error = 0
pair = (0, 0)

for i in range(10):     # 真實數字
    for j in range(10): # 預測數字
        if i != j:      # 排除掉對角線 (判斷正確的情況)
            if conf_matrix[i, j] > max_error:
                max_error = conf_matrix[i, j]
                pair = (i, j)

print(f"誤判次數最多的組合是：將 {pair[0]} 誤判為 {pair[1]}，共計 {max_error} 次。")

Max combin: 誤判次數最多的組合是：將 4 誤判為 9，共計 11 次。
